# 04. Conditioning: token/pooled text, clean audio/video, masks и cache

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Раскроем все входы условия, исполним encode_audio_condition/encode_video_condition с подписанными codec doubles и проверим собственно text-conditioning путь native DiT.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lecture 3B, notes §5 pp.34–40; Lab 3 Q2.2 (27–31), Q2.3 (32–33), Sanity Check 2.4 (34–36). Token/pooled encoders и modality codecs — промышленное расширение class-conditioning.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='04_conditioning_text_and_clean_modalities';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Условная плотность и три маршрута информации

Цель — p(x|c), neural field uθ(x,σ,c). Условие может входить через token context, глобальный modulation vector и clean modality/reference state. Это разные computational edges.

$$
T_v=P_v(T_{enc}),\quad T_a=P_a(T_{enc}),\qquad
c_v=\mathrm{Time}_v(1000\sigma_v)+P_{pool,v}(e_{pool}),
$$

$$
c_a=\mathrm{Time}_a(1000\sigma_a)+P_{pool,a}(e_{pool}).
$$

Qwen token features и pooled CLIP features — outputs отдельных pretrained encoders; DiT содержит собственные projections/refiners. Заморозка внешнего encoder не означает заморозку text projection внутри DiT. В tiny опытах features synthetic, vocabulary/tokenizer не загружаются; поэтому semantic prompt quality не измеряется.

Conditioning encoders ниже не делают transfer к device сами: caller обязан подготовить input на устройстве codec. Video helper переводит uint8 RGB в [-1,1], берёт **posterior.mode**, масштабирует и переставляет BCTHW→BTHWC, затем BF16. Audio helper требует mono (N,), делает B,D,A→B,A,D и scale.


In [ ]:
from types import SimpleNamespace
cond=native('core.algo.conditioning')
class VideoCodecDouble(nn.Module):
    def __init__(self):
        super().__init__();self.anchor=nn.Parameter(torch.tensor(1.));self.config={'scaling_factor':.5}
    def encode(self,pixels):
        # Coordinate/scale contract only: NOT learned Hunyuan VAE.
        z=F.avg_pool3d(pixels.mean(1,keepdim=True),kernel_size=(1,8,8))[:,:,::4].repeat(1,2,1,1,1)
        return SimpleNamespace(latent_dist=SimpleNamespace(mode=lambda:z))
class AudioCodecDouble:
    scaling_factor=.25
    def wrapped_encode(self,w):return w.unfold(-1,4,4).mean(-1)[:,None,:].repeat(1,4,1)
white=torch.full((1,3,9,16,16),255,dtype=torch.uint8)
vc=cond.encode_video_condition(white,VideoCodecDouble())
wave=torch.ones(16,requires_grad=True);ac=cond.encode_audio_condition(wave,AudioCodecDouble())
assert vc.shape==(1,3,2,2,2) and vc.dtype==torch.bfloat16 and vc.eq(.5).all()
assert ac.shape==(1,4,4) and ac.eq(.25).all() and not ac.requires_grad
metrics['codec_contracts']={'video':list(vc.shape),'audio':list(ac.shape),'dtype':str(vc.dtype),'grad_disabled':not ac.requires_grad}
for fn,arg,codec in [(cond.encode_audio_condition,torch.ones(2,16),AudioCodecDouble()),(cond.encode_video_condition,white.float(),VideoCodecDouble())]:
    try:fn(arg,codec)
    except ValueError:pass
    else:raise AssertionError('Invalid conditioning input should be rejected')
show_source('core/algo/conditioning.py')


## 2. Mask polarity: boolean keep-mask не равна float 0/1 bias

При masked softmax запрещённые keys должны иметь −∞ в additive mask. PyTorch SDPA boolean mask использует True=разрешено. Float mask складывается со scores; float 0/1 ничего не запрещает.

$$
M_j=\begin{cases}0,&\mathrm{valid}_j,\\-\infty,&\mathrm{padding}_j.\end{cases}
$$

`_normalize_attn_mask` в K6 меняет shape B,S→B,1,1,S, но **не преобразует float 0/1 в bool**. Контракт caller должен определять dtype и семантику. Проверьте хотя бы один valid key на пример, особенно в generic softmax implementation.


In [ ]:
ca=native().MultiheadCrossAttention(24,8,engine='sdpa',text_token_padding=True)
q=torch.randn(1,3,24);text=torch.randn(1,4,24);mask=torch.tensor([[[[True,True,True,False]]]])
changed=text.clone();changed[:,-1]+=100
with torch.no_grad():
    base=ca(q,text,mask);correct=ca(q,changed,mask);wrong=ca(q,changed,mask.float())
assert torch.equal(base,correct)
metrics['bool_mask_error']=float((base-correct).abs().max())
metrics['float_01_mask_change']=float((base-wrong).abs().max())
assert metrics['float_01_mask_change']>1e-4


## 3. Что можно кэшировать

Token/pooled **projections** не зависят от σ и могут быть переиспользованы внутри одной генерации, если inputs и weights неизменны. LTF text blocks используют time-conditioned modulation и пересчитываются на каждом time.

В source key cache использует data_ptr и shape. In-place изменение того же tensor не меняет data_ptr; caller должен считать conditioning immutable или вызывать clear_text_proj_cache. При training нельзя переиспользовать cached autograd graph между backward. Мы очищаем cache перед training forward.

Проверим вызовы projection и refiner hooks. Затем намеренно изменим text tensor in-place и сравним stale cache с clear; это иллюстрация контракта cache, не blanket bug claim для штатного immutable inference.


In [ ]:
model=tiny_model().eval();activate_gates(model);_,_,kw=tiny_inputs(model)
counts={'projection':0,'refiner':0}
def inc(name):
    def hook(*args):counts[name]+=1
    return hook
h1=model.video_text_embeddings.register_forward_hook(inc('projection'))
h2=model.video_text_transformer_blocks[0].register_forward_hook(inc('refiner'))
model.clear_text_proj_cache()
with torch.no_grad():
    first=model(**kw)
    later=dict(kw,time=torch.tensor([300.]))
    second=model(**later)
h1.remove();h2.remove()
assert counts=={'projection':1,'refiner':2}
with torch.no_grad():
    kw['text_embed'].add_(.5*torch.randn_like(kw['text_embed']))
    stale=model(**later)
    model.clear_text_proj_cache();fresh=model(**later)
metrics['cache_hook_counts']=counts
metrics['mutable_input_cache_difference']=float((stale[0]-fresh[0]).square().mean())
assert metrics['mutable_input_cache_difference']>0
print(metrics['cache_hook_counts'])


## 4. Null conditions и wrapper boundaries

`_resolve_null_embeds` выбирает shared null или separate video/audio nulls. При отсутствии audio null/RoPE делает documented fallback на video null. Это branch input подготовка; CFG algebra будет в lab 06. `encode_*_condition` использует no_grad: подготовка clean condition и обучение codec — разные стадии.


In [ ]:
loop=native('core.algo.denoise_loop');pv=dummy_text(positive=0.);pa=dummy_text(positive=-1.)
r1=torch.zeros(3,1,4,2,2);r2=torch.ones_like(r1)
te,pe,rr=loop._resolve_null_embeds([pv,pa],[r1,r2]);assert isinstance(te,list) and len(te)==2
te2,pe2,rr2=loop._resolve_null_embeds([pv,None],[r1,None]);assert te2 is pv['text_embeds']
metrics['null_branches']={'separate_modalities':len(te),'fallback_shape':list(te2.shape)}


## Измерение → вывод

Native conditioning helpers прошли scale/layout/validation проверки на явных codec doubles. Projection cache отделён от time-dependent refiners; masks и immutable-input предпосылка проверены. Семантические encoders и реальный codec не запускались.


## Самостоятельные опыты

1. Вернитесь в MIT Lab 3 Q2.2: где возникает unconditional branch при обучении?
2. Сравните mode/sample clean reference при одном seed и сформулируйте preprocessing contract.
3. Проверьте cache после weight update, device/dtype change и нового prompt.
4. Почему float mask нельзя исправить только reshape?

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
